# Linked List

*Singly · Doubly · LRU Cache · Cycle Detection · Real-World*



---
## 🧠 Mental Model: Linked Lists

> **A linked list is a chain of nodes. Each node knows only its value and its next neighbour — there's no index, no jumping ahead.**  
> The fundamental trade-off: O(1) insert/delete at a *known position* vs O(n) to *find* any position.

### WHY — Why does it exist?
Arrays (Python lists) give O(1) random access but O(n) insert/delete in the middle. Linked lists flip this: O(1) insert/delete (given a pointer to the node) but O(n) access by index. They're the backbone of LRU caches, job queues, and adjacency lists.

### WHAT — Variants

| Type | Structure | Extra cost | Best for |
|------|-----------|-----------|----------|
| Singly linked | `node → node → None` | O(n) delete (need prev) | Stacks, forward-only traversal |
| Doubly linked | `None ← node ↔ node → None` | 2× memory | O(1) delete with node ptr (LRU cache) |
| Circular | last → first | O(1) tail access | Round-robin, ring buffer |

### HOW — Key pointer diagrams

**Reversal (in-place, 3 pointers):**
```
Initial:   prev=None  curr=1→2→3→None
Step 1:    save next=2; 1→None; prev=1; curr=2
Step 2:    save next=3; 2→1; prev=2; curr=3
Step 3:    save next=None; 3→2; prev=3; curr=None
Result:    3→2→1→None
```

**Floyd's cycle detection (two pointers):**
```
slow moves 1 step, fast moves 2 steps
If they ever meet → cycle exists
If fast reaches None → no cycle
```

**Sentinel (dummy) node pattern:**
```python
dummy = ListNode(0)
dummy.next = head    # no special-case for empty list
# ... all operations work uniformly ...
return dummy.next
```

### WHEN — Use cases

| Scenario | Use |
|----------|-----|
| LRU cache (O(1) move-to-front) | Doubly linked + hash map |
| Undo/redo history | Doubly linked list |
| Task queue (in custom structures) | Singly linked |
| Merge K sorted lists | Linked list merge + min-heap |
| Detecting cycle in sequence | Floyd's two-pointer |

**WHEN NOT:** For general queues/stacks in Python, use `collections.deque` — it's a C-level doubly-linked list and is faster than any pure-Python implementation.

**Top 5 gotchas:**
1. **Off-by-one** — to delete node at index i, you need a pointer to i-1
2. **Head overwrites** — save `curr.next` before relinking `curr.next = new`
3. **Floyd's must start both pointers at head** — not one at head, one at second
4. **Reversal needs 3 pointers** (prev, curr, next_node), not 2
5. **Sentinel/dummy node** eliminates all head-special-cases; always use it

```
Complexity:
  Access by index:  O(n)
  Search:           O(n)
  Insert (at head): O(1)
  Insert (at tail): O(1) with tail pointer, else O(n)
  Delete (known node): O(1) doubly-linked; O(n) singly-linked
  Space:            O(n)
```


A linked list is a chain of nodes where each node holds a value AND a
pointer to the next node.  There is NO random access (unlike an array).
O(1) insert/delete at a KNOWN position;  O(n) to FIND the position.

When to use a linked list over a list/deque:
  ✓ You're building from scratch a data structure that needs O(1)
    splicing in the middle WITH a reference to the node (e.g. LRU cache)
  ✗ For general-purpose queues prefer collections.deque (C-level, faster)
  ✗ For stacks prefer list.append/pop (amortised O(1), cache-friendly)

> ⚠️ **GOTCHA 1: Off-by-one in traversal / deletion — the hardest class of bug.**
> ⚠️ **GOTCHA 2: Losing the list by overwriting `head` before saving `head.next`.**
> ⚠️ **GOTCHA 3: Cycle detection — Floyd's two-pointer MUST start both at head.**
> ⚠️ **GOTCHA 4: Reversing in-place — need THREE pointers (prev, curr, next_node).**
> ⚠️ **GOTCHA 5: Sentinel nodes eliminate most null-check edge cases in doubly-LL.**

In [ ]:
def notebook_linked_list_gotchas() -> None:

## §1 · Linked List Gotchas

In [ ]:
# ── §1.1  Off-by-one in deletion ─────────────────────────────────────
    #
    # To DELETE a node at index i, you need a pointer to node at index i-1.
    # GOTCHA: if you advance `curr` to i before saving `curr.prev`, you lose
    # the link you need to re-attach.

    head = build_list([1, 2, 3, 4, 5])

    def delete_nth(head, n):
        """Delete node at 0-based index n.  Returns new head."""
        if n == 0:
            return head.next                     # special-case: remove head
        prev, curr = None, head
        for _ in range(n):
            prev, curr = curr, curr.next         # advance BOTH together
        prev.next = curr.next                    # skip over curr
        return head

    head = delete_nth(head, 2)   # remove index 2 (value 3)
    assert to_list(head) == [1, 2, 4, 5], to_list(head)
    print("Delete nth node: [1,2,3,4,5] remove idx-2 →", to_list(head))

    # ── §1.2  Reversing in-place — three-pointer technique ───────────────
    #
    # GOTCHA: you MUST save `curr.next` BEFORE overwriting `curr.next = prev`.
    # If you forget, you lose the rest of the list.

    head2 = build_list([1, 2, 3, 4, 5])
    # Wrong intuition: swap pairs iteratively without saving next_node
    def reverse_correct(h):
        prev, curr = None, h
        while curr:
            next_node = curr.next   # SAVE before overwrite
            curr.next = prev        # reverse the pointer
            prev, curr = curr, next_node
        return prev                 # prev is new head

    rev = reverse_correct(head2)
    assert to_list(rev) == [5, 4, 3, 2, 1]
    print("Reverse in-place: [1,2,3,4,5] →", to_list(rev))

    # ── §1.3  Floyd's cycle detection — why both start at head ───────────
    #
    # GOTCHA: if fast starts one step ahead, the algorithm can miss a cycle
    # of length 1 (single-node self-loop).  Always start BOTH at head.

    # Build a cycle: 1 → 2 → 3 → 4 → 2 (back to node 2)
    n1, n2, n3, n4 = Node(1), Node(2), Node(3), Node(4)
    n1.next, n2.next, n3.next, n4.next = n2, n3, n4, n2   # cycle!

    assert has_cycle(n1) is True
    assert has_cycle(build_list([1, 2, 3])) is False
    print("Floyd's cycle detection: cycle? True / no-cycle? False ✓")

    # ── §1.4  Sentinel nodes eliminate edge cases ─────────────────────────
    #
    # Mental model: use a "dummy" head node before the real head.
    # Then EVERY deletion is a "delete a non-head node" — no special-casing!

    class SentinelList:
        class Node:
            def __init__(self, v): self.v = v; self.next = None

        def __init__(self):
            self.dummy = self.Node(-1)   # sentinel — never removed
            self.dummy.next = None

        def prepend(self, v):
            n = self.Node(v)
            n.next = self.dummy.next
            self.dummy.next = n

        def delete_value(self, v):
            prev, curr = self.dummy, self.dummy.next
            while curr:
                if curr.v == v:
                    prev.next = curr.next   # uniform: no head special-case
                    return
                prev, curr = curr, curr.next

        def to_list(self):
            out, cur = [], self.dummy.next
            while cur:
                out.append(cur.v); cur = cur.next
            return out

    sl = SentinelList()
    for v in [5, 4, 3, 2, 1]: sl.prepend(v)
    sl.delete_value(1)   # delete head (no special case!)
    sl.delete_value(3)   # delete middle
    sl.delete_value(5)   # delete tail
    print("Sentinel list after deletes:", sl.to_list())   # [2, 4]

    # ── §1.5  When NOT to use a linked list ───────────────────────────────
    #
    # GOTCHA: Python linked lists are MUCH slower than deque/list in practice
    # because:
    #   1. Each Node is a separate heap allocation (Python object overhead ~56B)
    #   2. Pointer chasing kills CPU cache (cache miss per node)
    #   3. deque uses fixed-size blocks of pointers → far better cache locality

    n = 10_000
    ll = build_list(range(n))      # pure Python linked list

    t0 = time.perf_counter()
    cur = ll
    while cur: cur = cur.next
    t_ll = time.perf_counter() - t0

    dq = deque(range(n))
    t0 = time.perf_counter()
    for _ in dq: pass
    t_dq = time.perf_counter() - t0

    print(f"\nTraversal of {n} elements:")
    print(f"  Custom linked list: {t_ll*1000:.2f}ms")
    print(f"  collections.deque:  {t_dq*1000:.2f}ms  ← use this in production!")

LRU = Least-Recently-Used eviction.
Data structure = doubly-linked list (order) + hash map (O(1) lookup).
  • get(key)  → O(1): hash map lookup + move node to front
  • put(key)  → O(1): hash map insert + remove LRU tail if over capacity

Why doubly linked (not singly)?
  Eviction from the tail requires O(1) node removal.
  To remove a node, you need BOTH prev and next pointers.
  With singly-linked: you'd need O(n) to find the predecessor of the tail.

> ⚠️ **GOTCHA 1: The sentinel head/tail trick — eliminates head==None edge cases.**
> ⚠️ **GOTCHA 2: When capacity is 1, put(existing_key) must still move to front.**
> ⚠️ **GOTCHA 3: Thread safety — LRUCache is NOT thread-safe; add a lock for concurrency.**
> ⚠️ **GOTCHA 4: In production use functools.lru_cache or cachetools.LRUCache.**

In [ ]:
def notebook_lru_gotchas() -> None:

## §2 · LRU Cache Gotchas

In [ ]:
# ── §2.1  Basic correctness ───────────────────────────────────────────
    cache = LRUCache(3)
    cache.put(1, 10); cache.put(2, 20); cache.put(3, 30)
    assert cache.get(1) == 10     # touches 1 → now MRU
    cache.put(4, 40)              # evicts 2 (LRU after 1 was touched)
    assert cache.get(2) == -1     # evicted ✓
    assert cache.get(3) == 30     # still present
    print("LRU basic correctness ✓")

    # ── §2.2  GOTCHA: update existing key must re-order ───────────────────
    cache2 = LRUCache(2)
    cache2.put(1, 1); cache2.put(2, 2)
    cache2.put(1, 100)            # UPDATE existing key 1 → must move to MRU
    cache2.put(3, 3)              # evicts 2 (LRU), NOT 1
    assert cache2.get(1) == 100   # 1 must still be here ✓
    assert cache2.get(2) == -1    # 2 was evicted ✓
    print("LRU update re-orders correctly ✓")

    # ── §2.3  GOTCHA: capacity 1 edge case ───────────────────────────────
    cache3 = LRUCache(1)
    cache3.put(1, 1)
    cache3.put(2, 2)              # evicts 1
    assert cache3.get(1) == -1   # evicted
    assert cache3.get(2) == 2    # present
    print("LRU capacity=1 edge case ✓")

    # ── §2.4  Production alternatives ─────────────────────────────────────
    from functools import lru_cache

    @lru_cache(maxsize=128)
    def expensive(n):
        return n * n

    _ = [expensive(i) for i in range(200)]
    info = expensive.cache_info()
    print(f"\nfunctools.lru_cache info: {info}")
    print("  Use functools.lru_cache for pure functions — C-level, thread-safe")

ShopFlow caches product page HTML. When a product is viewed, it moves
to the "most recently used" position. When the cache fills, the least
recently used product is evicted first. This LRU behaviour requires
O(1) get AND O(1) eviction — impossible with a plain dict or list.
The solution: a doubly-linked list (for O(1) node reorder) + a hash
map (for O(1) key lookup). This is exactly what Python's functools.lru_cache
uses internally.

In [ ]:
def scenario_linked_list_lru() -> None:

In [ ]:
h("WHY a doubly-linked list? Why not a simple list?")
    print("""
  Operation needed      | list          | doubly-linked list
  ──────────────────────┼───────────────┼──────────────────────
  Remove node by ref    | O(n) — search | O(1) — use prev/next
  Move node to front    | O(n)          | O(1) — re-link 3 ptrs
  Remove least-recent   | O(1) tail pop | O(1) — tail sentinel
  Lookup by key         | O(n)          | O(1) via hash map

  Total LRU operation:
    list:  O(n) per get/put  — at 10k cached products: 10k comparisons!
    LL+map: O(1) per get/put — always constant, regardless of cache size
""")

    h("LRU Cache implementation (same structure as functools.lru_cache)")
    from dataclasses import dataclass

    @dataclass
    class _Node:
        key:  str = ""
        val:  object = None
        prev: "_Node | None" = None
        next: "_Node | None" = None

    class LRUProductCache:
        def __init__(self, capacity: int) -> None:
            self._cap  = capacity
            self._map: dict[str, _Node] = {}
            # Sentinel head (MRU side) and tail (LRU side) — eliminates null checks
            self._head, self._tail = _Node(), _Node()
            self._head.next = self._tail
            self._tail.prev = self._head

        def _remove(self, node: _Node) -> None:
            node.prev.next = node.next   # type: ignore[union-attr]
            node.next.prev = node.prev   # type: ignore[union-attr]

        def _push_front(self, node: _Node) -> None:
            node.prev = self._head
            node.next = self._head.next
            self._head.next.prev = node  # type: ignore[union-attr]
            self._head.next      = node

        def get(self, key: str) -> object:
            if key not in self._map: return -1
            node = self._map[key]
            self._remove(node)        # unlink from current position
            self._push_front(node)    # move to MRU (head side)
            return node.val

        def put(self, key: str, val: object) -> None:
            if key in self._map:
                self._remove(self._map[key])
            node = _Node(key, val)
            self._map[key] = node
            self._push_front(node)
            if len(self._map) > self._cap:   # evict LRU
                lru = self._tail.prev        # node just before tail sentinel
                self._remove(lru)            # type: ignore[arg-type]
                del self._map[lru.key]       # type: ignore[union-attr]

        def order(self) -> list[str]:        # MRU → LRU for debugging
            result, cur = [], self._head.next
            while cur is not self._tail:
                result.append(cur.key)  # type: ignore[union-attr]
                cur = cur.next           # type: ignore[assignment]
            return result

    cache = LRUProductCache(3)
    cache.put("shoes_001", {"name": "Running Shoes", "price": 89.99})
    cache.put("shirt_002", {"name": "Cotton Shirt",  "price": 29.99})
    cache.put("pants_003", {"name": "Slim Pants",    "price": 59.99})
    print(f"After 3 inserts (MRU→LRU): {cache.order()}")

    cache.get("shoes_001")    # user views shoes — moves to MRU
    print(f"After viewing shoes:        {cache.order()}")

    cache.put("hat_004", {"name": "Baseball Cap", "price": 19.99})   # evicts LRU
    print(f"After adding hat (shirt evicted): {cache.order()}")
    assert cache.get("shirt_002") == -1   # evicted!

    h("In production: use Python's built-in lru_cache")
    @lru_cache(maxsize=1000)
    def get_product_html(product_id: str) -> str:
        # Simulates expensive DB + template render (200ms in production)
        return f"<html>Product {product_id}</html>"

    get_product_html("shoes_001")
    get_product_html("shoes_001")   # cache hit
    info = get_product_html.cache_info()
    print(f"\nfunctools.lru_cache: {info}")
    print("  For method-level caching use cachetools.LRUCache (not lru_cache)")

    h("WHERE in real systems")
    print("""
  Redis: LRU is the default eviction policy (maxmemory-policy=allkeys-lru)
    Every key access updates the LRU clock — same conceptual structure
  Python functools.lru_cache: uses a doubly-linked list + dict, as above
  Django cache framework: per-backend LRU (LocMemCache uses OrderedDict trick)
  HTTP browsers: page cache is LRU by tab/domain policy
  CPU L1/L2 caches: hardware LRU eviction of memory lines
""")